# Simple scFoundation Masked Gene Prediction

Minimal notebook to:
1. read an AnnData file
2. use scFoundation's gene selection function
3. run masked gene prediction inference

In [ ]:
from pathlib import Path
import sys

import anndata as ad
import numpy as np
import pandas as pd
import torch
from scipy.sparse import issparse

REPO_ROOT = Path.cwd()
SCF_MODEL_DIR = REPO_ROOT / "external" / "scFoundation" / "model"
CKPT_PATH = SCF_MODEL_DIR / "models.ckpt"
ADATA_PATH = REPO_ROOT / "path/to/your_data.h5ad"
COUNTS_SOURCE = "raw"  # or "X"
N_CELLS = 8
MASK_FRACTION = 0.3
RANDOM_SEED = 0

if str(SCF_MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(SCF_MODEL_DIR))

from load import getEncoerDecoderData, load_model_frommmf, main_gene_selection

In [ ]:
adata = ad.read_h5ad(ADATA_PATH)
if COUNTS_SOURCE == "raw":
    if adata.raw is None:
        raise ValueError("COUNTS_SOURCE='raw' but adata.raw is missing")
    matrix = adata.raw.X
    var = adata.raw.var.copy()
else:
    matrix = adata.X
    var = adata.var.copy()

gene_names = var["gene_name"].astype(str).tolist() if "gene_name" in var.columns else var.index.astype(str).tolist()
matrix = matrix[:N_CELLS]
counts = matrix.toarray() if issparse(matrix) else np.asarray(matrix)
counts_df = pd.DataFrame(counts, index=adata.obs_names[:N_CELLS], columns=gene_names)
counts_df.head()

In [ ]:
gene_panel = pd.read_csv(SCF_MODEL_DIR / "OS_scRNA_gene_index.19264.tsv", sep="\t")
gene_list = gene_panel["gene_name"].astype(str).tolist()
selected_df, to_fill_columns, selected_var = main_gene_selection(counts_df, gene_list)
selected_df.shape, len(to_fill_columns)

In [ ]:
model, config = load_model_frommmf(str(CKPT_PATH), "gene")
model.eval()

counts_19264 = selected_df.to_numpy(dtype=np.float32)
total_counts = counts_19264.sum(axis=1)
normalized = np.log1p((counts_19264 / np.clip(total_counts[:, None], 1.0, None)) * 1e4).astype(np.float32)

# simplest choice: S = T = log10(total_counts)
s_token = np.log10(np.clip(total_counts, 1.0, None)).astype(np.float32)
seq = np.concatenate([normalized, s_token[:, None], s_token[:, None]], axis=1).astype(np.float32)

rng = np.random.default_rng(RANDOM_SEED)
zero_padded = selected_var["mask"].to_numpy(dtype=bool)
gene_mask = np.zeros_like(counts_19264, dtype=bool)
valid_candidates = (counts_19264 > 0) & (~zero_padded[None, :])

for i in range(counts_19264.shape[0]):
    candidates = np.flatnonzero(valid_candidates[i])
    if candidates.size == 0:
        continue
    n_mask = max(1, int(np.floor(candidates.size * MASK_FRACTION)))
    chosen = candidates if n_mask >= candidates.size else rng.choice(candidates, size=n_mask, replace=False)
    gene_mask[i, chosen] = True

seq_masked = seq.copy()
seq_raw = seq.copy()
seq_masked[:, :19264][gene_mask] = float(config["mask_token_id"])
seq_raw[:, :19264][gene_mask] = 0.0

In [ ]:
with torch.no_grad():
    x = torch.from_numpy(seq_masked).to(device="cuda", dtype=torch.float32)
    x_raw = torch.from_numpy(seq_raw).to(device="cuda", dtype=torch.float32)
    (
        encoder_data,
        encoder_position_gene_ids,
        encoder_data_padding,
        encoder_labels,
        decoder_data,
        decoder_data_padding,
        _,
        _,
        decoder_position_gene_ids,
    ) = getEncoerDecoderData(x, x_raw, config)

    prediction = model.forward(
        x=encoder_data,
        padding_label=encoder_data_padding,
        encoder_position_gene_ids=encoder_position_gene_ids,
        encoder_labels=encoder_labels,
        decoder_data=decoder_data,
        mask_gene_name=False,
        mask_labels=None,
        decoder_position_gene_ids=decoder_position_gene_ids,
        decoder_data_padding_labels=decoder_data_padding,
    )

prediction = prediction[:, :19264].detach().cpu().numpy()
masked_mse = ((prediction - normalized) ** 2 * gene_mask).sum(axis=1) / np.clip(gene_mask.sum(axis=1), 1, None)
results = pd.DataFrame({
    "cell_id": selected_df.index,
    "masked_gene_count": gene_mask.sum(axis=1),
    "masked_mse": masked_mse,
})
results